# DynaCLR 数据查看

论文: https://arxiv.org/abs/2410.11281v2

本notebook展示DynaCLR三类数据的查看方法：ALFI细胞周期、Microglia/DynaMorph、DynaCLR感染演示。

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

base = Path('..')
print(f'项目目录: {base.resolve()}')

# 加载数据清单
alfi = pd.read_csv(base / 'manifests' / 'alfi_cellcycle.csv')
dynamorph = pd.read_csv(base / 'manifests' / 'microglia_dynamorph.csv')
infection = pd.read_csv(base / 'manifests' / 'dynaclr_infection.csv')
software = pd.read_csv(base / 'manifests' / 'software_versions.csv')

print(f'\n=== 数据清单 ===')
print(f'ALFI细胞周期: {len(alfi)} 个文件记录')
print(f'Microglia/DynaMorph: {len(dynamorph)} 个文件记录')
print(f'DynaCLR感染: {len(infection)} 个文件记录')
print(f'软件版本: {len(software)} 个')

## 软件版本

In [ ]:
print(software[['software_name','package_name','purpose','repo_url']].to_string(index=False))
print('\n注意: 核对论文v2对应的代码版本与配置，不能将最新仓库默认配置当成原论文配置。')

## 查看 ALFI 细胞周期数据

In [ ]:
alfi_dir = base / 'data' / 'alfi_cellcycle'
tif_files = list(alfi_dir.glob('*.tif*')) + list(alfi_dir.glob('*.nd2'))
print(f'ALFI图像文件: {len(tif_files)}')

if tif_files:
    from tifffile import TiffFile
    f = tif_files[0]
    print(f'\n查看: {f.name} ({f.stat().st_size/1024/1024:.1f} MB)')
    with TiffFile(f) as tif:
        print(f'页数: {len(tif.pages)}')
        arr = tif.pages[0].asarray()
        print(f'第一页: shape={arr.shape}, dtype={arr.dtype}')
        print(f'值范围: [{arr.min()}, {arr.max()}]')
else:
    print('未找到图像文件。请先运行 download_alfi.py --list-only 查看可用文件。')
    print('ALFI数据来源: https://doi.org/10.6084/m9.figshare.23798451')

## 查看追踪文件

In [ ]:
tracking_files = list(alfi_dir.glob('*track*.csv')) + list(alfi_dir.glob('*tracking*.csv'))
if tracking_files:
    df = pd.read_csv(tracking_files[0])
    print(f'追踪文件: {tracking_files[0].name}')
    print(f'形状: {df.shape}')
    print(f'列: {list(df.columns)}')
    if 'track_id' in df.columns:
        print(f'轨迹数: {df["track_id"].nunique()}')
    if 'frame' in df.columns:
        print(f'帧范围: {df["frame"].min()} - {df["frame"].max()}')
else:
    print('未找到追踪文件。')
    print('追踪文件应包含: track_id, frame, parent_id, x, y, label等字段')

## 数据类别区分提醒

In [ ]:
print('=== 重要提醒 ===')
print('1. 必须区分完整训练数据、测试数据、示例数据、模型权重和演示视频')
print('2. 不能因为下载了示例包，就报告已获得论文全部数据')
print('3. 人工标签、模型预测标签和伪标签必须分开记录')
print('4. 登革病毒与Zika数据不得未经核对混合')
print('5. 不默认转换成单细胞转录组使用的h5ad')
print('6. 使用适合多维图像的读取方式，避免把全部大型图像一次性加载进内存')
print('7. 代码许可证不代表数据许可证')